<a href="https://colab.research.google.com/github/3295354473-art/5fold-cv-project/blob/main/NGAFID_DATASET_TF_EXAMPLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INSTALL PREREQ

In [2]:
!git clone https://github.com/hyang0129/NGAFIDDATASET.git

!(cd NGAFIDDATASET ; git checkout main; git reset --hard HEAD; git pull)
!(cd NGAFIDDATASET ; pip install -r requirements.txt -q)



fatal: destination path 'NGAFIDDATASET' already exists and is not an empty directory.
Already on 'main'
Your branch is up to date with 'origin/main'.
HEAD is now at fa72386 Update README.md
Already up to date.


# IMPORT DEPENDENCIES

In [3]:
import sys
sys.path.append('/content/NGAFIDDATASET')

# Install the ngafiddataset package from the cloned repository
# This ensures the package and its submodules are properly discoverable


In [4]:
from tqdm.autonotebook import tqdm


In [5]:
from ngafiddataset.dataset.dataset import NGAFID_Dataset_Manager
from ngafiddataset.dataset.utils import to_dict_of_list
from ngafiddataset.utils import connect_to_tpu
import pandas as pd


strategy = connect_to_tpu()

No TPU Available
REPLICAS:  1


# DEFINE MODEL FN

In [15]:
import tensorflow as tf

tfk = tf.keras
tfkl = tf.keras.layers

class Classifier_INCEPTION:
    def __init__(
        self,
        input_shape,
        nb_classes,
        build=True,
        batch_size=64,
        nb_filters=32,
        use_residual=True,
        use_bottleneck=True,
        depth=6,
        kernel_size=41,
        nb_epochs=1500,
        two_output = False,
        mode = None
    ):

        self.nb_filters = nb_filters
        self.use_residual = use_residual
        self.use_bottleneck = use_bottleneck
        self.depth = depth
        self.kernel_size = kernel_size - 1
        self.callbacks = None
        self.batch_size = batch_size
        self.bottleneck_size = 32
        self.nb_epochs = nb_epochs
        self.two_output = two_output
        self.mode = mode

        if build is True:
            self.model = self.build_model(input_shape, nb_classes)

    def _inception_module(self, input_tensor, stride=1, activation="linear"):

        if self.use_bottleneck and int(input_tensor.shape[-1]) > 1:
            input_inception = tf.keras.layers.Conv1D(
                filters=self.bottleneck_size, kernel_size=1, padding="same", activation=activation, use_bias=False
            )(input_tensor)
        else:
            input_inception = input_tensor

        # kernel_size_s = [3, 5, 8, 11, 17]
        kernel_size_s = [self.kernel_size // (2 ** i) for i in range(3)]

        conv_list = []

        for i in range(len(kernel_size_s)):
            conv_list.append(
                tf.keras.layers.Conv1D(
                    filters=self.nb_filters,
                    kernel_size=kernel_size_s[i],
                    strides=stride,
                    padding="same",
                    activation=activation,
                    use_bias=False,
                )(input_inception)
            )

        max_pool_1 = tf.keras.layers.MaxPool1D(pool_size=3, strides=stride, padding="same")(input_tensor)

        conv_6 = tf.keras.layers.Conv1D(
            filters=self.nb_filters, kernel_size=1, padding="same", activation=activation, use_bias=False
        )(max_pool_1)

        conv_list.append(conv_6)

        x = tf.keras.layers.Concatenate(axis=2)(conv_list)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation(activation="relu")(x)
        return x

    def _shortcut_layer(self, input_tensor, out_tensor):
        shortcut_y = tf.keras.layers.Conv1D(
            filters=int(out_tensor.shape[-1]), kernel_size=1, padding="same", use_bias=False
        )(input_tensor)
        shortcut_y = tf.keras.layers.BatchNormalization()(shortcut_y)

        x = tf.keras.layers.Add()([shortcut_y, out_tensor])
        x = tf.keras.layers.Activation("relu")(x)
        return x

    def build_model(self, input_shape, nb_classes):
        input_layer = tf.keras.layers.Input(input_shape, name = 'data')

        x = input_layer
        input_res = input_layer

        for d in range(self.depth):

            x = self._inception_module(x)

            if self.use_residual and d % 3 == 2:
                x = self._shortcut_layer(input_res, x)
                input_res = x

        gap_layer = tf.keras.layers.GlobalAveragePooling1D()(x)


        outputs = []

        outputs.append(tf.keras.layers.Dense(nb_classes, activation="softmax", name='target_class')(gap_layer))

        if self.two_output:
            outputs.append(tf.keras.layers.Dense(1, activation="sigmoid", name='before_after')(gap_layer))


        if self.mode == 'before_after':
            outputs = []
            outputs.append(tf.keras.layers.Dense(1, activation="sigmoid", name='before_after')(gap_layer))

        model = tf.keras.models.Model(inputs=input_layer, outputs=outputs)

        return model

    def get_kernel_model(self):
        '''
        Get a model whose output is just the global average pooling layer.

        Returns:

        '''

        return tf.keras.Model(self.model.layers[0].input, self.model.layers[-2].output)




In [16]:
def point_wise_feed_forward_network(d_model, dff):
  return tf.keras.Sequential([
      tf.keras.layers.Dense(dff, activation='relu'),  # (batch_size, seq_len, dff)
      tf.keras.layers.Dense(d_model)  # (batch_size, seq_len, d_model)
  ])

def scaled_dot_product_attention(q, k, v, mask):
  """Calculate the attention weights.
  q, k, v must have matching leading dimensions.
  k, v must have matching penultimate dimension, i.e.: seq_len_k = seq_len_v.
  The mask has different shapes depending on its type(padding or look ahead)
  but it must be broadcastable for addition.

  Args:
    q: query shape == (..., seq_len_q, depth)
    k: key shape == (..., seq_len_k, depth)
    v: value shape == (..., seq_len_v, depth_v)
    mask: Float tensor with shape broadcastable
          to (..., seq_len_q, seq_len_k). Defaults to None.

  Returns:
    output, attention_weights
  """

  matmul_qk = tf.matmul(q, k, transpose_b=True)  # (..., seq_len_q, seq_len_k)

  # scale matmul_qk
  dk = tf.cast(tf.shape(k)[-1], tf.float32)
  scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

  # add the mask to the scaled tensor.
  if mask is not None:
    scaled_attention_logits += (mask * -1e9)

  # softmax is normalized on the last axis (seq_len_k) so that the scores
  # add up to 1.
  attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)  # (..., seq_len_q, seq_len_k)

  output = tf.matmul(attention_weights, v)  # (..., seq_len_q, depth_v)

  return output, attention_weights

class MultiHeadAttention(tf.keras.layers.Layer):
  def __init__(self, d_model, num_heads):
    super(MultiHeadAttention, self).__init__()
    self.num_heads = num_heads
    self.d_model = d_model

    assert d_model % self.num_heads == 0

    self.depth = d_model // self.num_heads

    self.wq = tf.keras.layers.Dense(d_model)
    self.wk = tf.keras.layers.Dense(d_model)
    self.wv = tf.keras.layers.Dense(d_model)

    self.dense = tf.keras.layers.Dense(d_model)

  def split_heads(self, x, batch_size):
    """Split the last dimension into (num_heads, depth).
    Transpose the result such that the shape is (batch_size, num_heads, seq_len, depth)
    """
    x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
    return tf.transpose(x, perm=[0, 2, 1, 3])

  def call(self, v, k, q, mask):
    batch_size = tf.shape(q)[0]

    q = self.wq(q)  # (batch_size, seq_len, d_model)
    k = self.wk(k)  # (batch_size, seq_len, d_model)
    v = self.wv(v)  # (batch_size, seq_len, d_model)

    q = self.split_heads(q, batch_size)  # (batch_size, num_heads, seq_len_q, depth)
    k = self.split_heads(k, batch_size)  # (batch_size, num_heads, seq_len_k, depth)
    v = self.split_heads(v, batch_size)  # (batch_size, num_heads, seq_len_v, depth)

    # scaled_attention.shape == (batch_size, num_heads, seq_len_q, depth)
    # attention_weights.shape == (batch_size, num_heads, seq_len_q, seq_len_k)
    scaled_attention, attention_weights = scaled_dot_product_attention(
        q, k, v, mask)

    scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])  # (batch_size, seq_len_q, num_heads, depth)

    concat_attention = tf.reshape(scaled_attention,
                                  (batch_size, -1, self.d_model))  # (batch_size, seq_len_q, d_model)

    output = self.dense(concat_attention)  # (batch_size, seq_len_q, d_model)

    return output, attention_weights

class EncoderLayer(tf.keras.layers.Layer):
  def __init__(self, d_model, num_heads, dff, rate=0.1):
    super(EncoderLayer, self).__init__()

    self.mha = MultiHeadAttention(d_model, num_heads)
    self.ffn = point_wise_feed_forward_network(d_model, dff)

    self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
    self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    self.dropout1 = tf.keras.layers.Dropout(rate)
    self.dropout2 = tf.keras.layers.Dropout(rate)

  def call(self, x, training=None, mask = None):

    attn_output, self.attention_values = self.mha(x, x, x, mask)  # (batch_size, input_seq_len, d_model)
    attn_output = self.dropout1(attn_output, training=training)
    out1 = self.layernorm1(x + attn_output)  # (batch_size, input_seq_len, d_model)

    ffn_output = self.ffn(out1)  # (batch_size, input_seq_len, d_model)
    ffn_output = self.dropout2(ffn_output, training=training)
    out2 = self.layernorm2(out1 + ffn_output)  # (batch_size, input_seq_len, d_model)

    return out2

def get_angles(pos, i, d_model):
    angle_rates = 1 / np.power(10000, (2 * (i//2)) / np.float32(d_model))
    return pos * angle_rates

def positional_encoding(position, d_model):
    angle_rads = get_angles(np.arange(position)[:, np.newaxis],
                            np.arange(d_model)[np.newaxis, :],
                            d_model)

    # apply sin to even indices in the array; 2i
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

    # apply cos to odd indices in the array; 2i+1
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    pos_encoding = angle_rads[np.newaxis, ...]

    return tf.cast(pos_encoding, dtype=tf.float32)

class PositionalEncoding(tf.keras.layers.Layer):

    def __init__(self, maximum_position_encoding = 1000):
        super(PositionalEncoding, self).__init__()

        self.pos_encoding = positional_encoding(maximum_position_encoding,
                                        512)

    def call(self, x):

        seq_len = tf.shape(x)[1]
        x += self.pos_encoding[:, :seq_len, :]

        return x

def get_mhsa_model(nbclass = 2, SHAPE = (4096, 23), mode = 'before_after'):
    # I realize that I did not include a positional embedding, but its too late now. Maybe a future paper could address this?
    d = 512
    ff = 1024
    model =  tfk.Sequential([tf.keras.Input(shape = SHAPE),
                                            tfkl.Conv1D(128, 7, strides = 1, padding='same', activation='relu'),
                                            # tfkl.BatchNormalization(),
                                            tfkl.Conv1D(128, 7, strides = 2, padding='same', activation='relu'),
                                            # tfkl.BatchNormalization(),
                                            tfkl.Conv1D(256, 7, strides = 1, padding='same', activation='relu'),
                                            # tfkl.BatchNormalization(),
                                            tfkl.Conv1D(256, 7, strides = 2, padding='same', activation='relu'),
                                            # tfkl.BatchNormalization(),
                                            # tfkl.Conv1D(512, 7, strides = 1, padding='same', activation='relu'),
                                            tfkl.Conv1D(512, 7, strides = 2, padding='same', activation='relu'),
                                            # tfkl.Conv1D(768, 7, strides = 1, padding='same', activation='relu'),
                                            # tfkl.BatchNormalization(),
                                            EncoderLayer(d_model=d, num_heads=8, dff=ff),
                                            EncoderLayer(d_model=d, num_heads=8, dff=ff),
                                            EncoderLayer(d_model=d, num_heads=8, dff=ff),
                                            EncoderLayer(d_model=d, num_heads=8, dff=ff),

                                            tf.keras.layers.GlobalAveragePooling1D(),
                                            # tfkl.Dense(nbclass, activation='softmax'),
    ])


    input = tf.keras.Input(shape = SHAPE, name = 'data')

    x = model(input)

    output = []
    if mode == 'before_after':
        output =  tfkl.Dense(1, activation='sigmoid', name = 'before_after')(x)
    elif mode == 'both':
        output.append(tf.keras.layers.Dense(nbclass, activation="softmax", name='target_class')(x))
        output.append(tfkl.Dense(1, activation='sigmoid', name = 'before_after')(x))
    elif mode == 'classes':
        output.append(tf.keras.layers.Dense(nbclass, activation="softmax", name='target_class')(x))

    fun_model = tf.keras.Model(input, output)

    return fun_model

def get_mhsa_model_pe():
    # I realize that I did not include a positional embedding, but its too late now. Maybe a future paper could address this?

    model =  tfk.Sequential([tf.keras.Input(shape = SHAPE),
                                            tfkl.Conv1D(128, 7, strides = 1, padding='same', activation='relu'),
                                            tfkl.BatchNormalization(),
                                            tfkl.Conv1D(128, 7, strides = 2, padding='same', activation='relu'),
                                            tfkl.BatchNormalization(),
                                            tfkl.Conv1D(256, 3, strides = 1, padding='same', activation='relu'),
                                            tfkl.BatchNormalization(),
                                            tfkl.Conv1D(256, 7, strides = 2, padding='same', activation='relu'),
                                            tfkl.BatchNormalization(),
                                            tfkl.Conv1D(512, 7, strides = 2, padding='same', activation='relu'),
                                            tfkl.BatchNormalization(),
                                            # tfkl.Conv1D(512, 7, strides = 2, padding='same', activation='relu'),
                                            # tfkl.Conv1D(512, 7, strides = 2, padding='same', activation='relu'),
                                            PositionalEncoding(),
                                            EncoderLayer(d_model=512, num_heads=8, dff=512),
                                            EncoderLayer(d_model=512, num_heads=8, dff=512),
                                            EncoderLayer(d_model=512, num_heads=8, dff=512),
                                            EncoderLayer(d_model=512, num_heads=8, dff=512),
                                            # tfkl.Lambda(lambda x : x[:, 0, :]),
                                            tf.keras.layers.GlobalAveragePooling1D(),
                                            tfkl.Dense(1, activation='sigmoid'),
    ])

    return model


model = get_mhsa_model(mode = 'both')
model.summary()


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ data (InputLayer)   │ (None, 4096, 23)  │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_4        │ (None, 512)       │ 10,153,344 │ data[0][0]        │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ target_class        │ (None, 2)         │      1,026 │ sequential_4[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ before_after        │ (None, 1)         │        513 │ sequential_4[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,154,883 (38.74 MB)

 Trainable params: 10,154,883 (38.74 MB)

 Non-trainable params: 0 (0.00 B)

# SETUP DATA

In [17]:
import os
import tarfile # Import tarfile for .tar files
import shutil # To remove existing partial downloads if any
import pandas as pd # Import pandas as it is used later

# Ensure NGAFID_Dataset_Manager is imported if it was part of this cell previously
# Assuming NGAFID_Dataset_Manager is already imported in a previous cell as per notebook state

dataset_name = '2days'
problematic_gdrive_id = '1-2pxwiQNhFnhTg7whosQoF_yztD5jOM2'
destination_base_path = '/content/'
expected_dataset_dir = os.path.join(destination_base_path, dataset_name)
downloaded_tar_path = os.path.join(destination_base_path, f'{dataset_name}.tar') # Changed to .tar

# --- BEGIN MODIFICATION FOR GOOGLE DRIVE ACCESS ---
# Assuming the user has uploaded '2days.tar' to their Google Drive 'My Drive/NGAFIDDATASET/'
gdrive_tar_path = '/content/drive/My Drive/NGAFIDDATASET/2days.tar' # Changed to .tar

if not os.path.exists(os.path.join(expected_dataset_dir, 'flight_header.csv')):
    print(f"Dataset '{dataset_name}' not found or incomplete at {expected_dataset_dir}.")

    if os.path.exists(gdrive_tar_path):
        print(f"Found dataset tar file in Google Drive: {gdrive_tar_path}. Attempting extraction...")
        # Clean up any partial previous downloads for the tar file
        if os.path.exists(downloaded_tar_path):
            os.remove(downloaded_tar_path)
        # Clean up the target directory if it exists but is incomplete
        if os.path.exists(expected_dataset_dir):
            if not os.path.exists(os.path.join(expected_dataset_dir, 'flight_header.csv')):
                print(f"Removing incomplete dataset directory: {expected_dataset_dir}")
                shutil.rmtree(expected_dataset_dir)

        os.makedirs(expected_dataset_dir, exist_ok=True)

        try:
            shutil.copy(gdrive_tar_path, downloaded_tar_path) # Copy from GDrive to local /content/
            with tarfile.open(downloaded_tar_path, 'r') as tar_ref: # Changed to tarfile.open
                tar_ref.extractall(destination_base_path) # Extracts to /content/2days/
            print(f"Extraction complete to {expected_dataset_dir}.")
            os.remove(downloaded_tar_path) # Optional: Remove the copied tar file after extraction
        except Exception as e:
            print(f"An error occurred during extraction from Google Drive. Please ensure the tar file is valid: {e}")
            raise # Re-raise if extraction fails
    else:
        # Original manual upload instructions if GDrive file is not found
        print("Automatic download failed. Please perform the following manual steps:")
        print("1. Open this URL in your browser to download the dataset: https://drive.google.com/uc?id=1-2pxwiQNhFnhTg7whosQoF_yztD5jOM2")
        print("2. Download the tar file to your local machine.")
        print(f"3. Upload the downloaded tar file directly to your Colab environment. Ensure it's named '{dataset_name}.tar' and located in '{destination_base_path}' (e.g., drag and drop to the file explorer on the left). OR ensure it is at '{gdrive_tar_path}'.")
        print("4. After uploading, rerun this cell. It will then attempt to extract the data.")

        # Clean up any partial previous downloads for the tar file
        if os.path.exists(downloaded_tar_path):
            os.remove(downloaded_tar_path)
        # Also clean up the target directory if it exists but is incomplete
        if os.path.exists(expected_dataset_dir):
            if not os.path.exists(os.path.join(expected_dataset_dir, 'flight_header.csv')):
                print(f"Removing incomplete dataset directory: {expected_dataset_dir}")
                shutil.rmtree(expected_dataset_dir)

        os.makedirs(expected_dataset_dir, exist_ok=True)

        if os.path.exists(downloaded_tar_path):
            print(f"'{downloaded_tar_path}' detected. Attempting extraction...")
            try:
                with tarfile.open(downloaded_tar_path, 'r') as tar_ref: # Changed to tarfile.open
                    tar_ref.extractall(destination_base_path) # Extracts to /content/2days/
                print(f"Extraction complete to {expected_dataset_dir}.")
                os.remove(downloaded_tar_path) # Optional: Remove the tar file after extraction
            except Exception as e:
                print(f"An error occurred during extraction. Please ensure the uploaded tar file is valid: {e}")
                raise # Re-raise if extraction fails
        else:
            # If the tar is still not found after instructions, raise an error to stop execution
            raise FileNotFoundError(f"Manual upload of '{gdrive_tar_path}' or '{downloaded_tar_path}' required. Please upload the file and rerun this cell.")
else:
    print(f"Dataset '{dataset_name}' found at {expected_dataset_dir}.")

# --- END MODIFICATION FOR GOOGLE DRIVE ACCESS ---

# --- Begin patch to prevent NGAFID_Dataset_Manager from trying to download again ---
# The dataset is already extracted, so we must prevent NGAFID_Dataset_Manager's internal download.

# Temporarily store the original download method
_original_download_method = NGAFID_Dataset_Manager.download

# Patch the download method to be a no-op if flight_header.csv already exists
def _patched_download_method(cls, name, destination='./', extract=True):
    if not os.path.exists(os.path.join(destination, name, 'flight_header.csv')):
        # Only call original download if flight_header.csv truly doesn't exist (unexpected at this point)
        print("NGAFID_Dataset_Manager's internal download method was called, but data should already be present. Attempting original download.")
        _original_download_method.__func__(cls, name, destination, extract) # Call original static method
    else:
        print("NGAFID_Dataset_Manager's internal download method bypassed as data exists.")

NGAFID_Dataset_Manager.download = classmethod(_patched_download_method)
# --- End patch ---

print(f"Initializing NGAFID_Dataset_Manager with dataset '{dataset_name}'...")
dm = NGAFID_Dataset_Manager(dataset_name, destination=destination_base_path)
df = pd.read_csv(os.path.join(expected_dataset_dir, 'flight_header.csv'))

dm.data_dict =  dm.construct_data_dictionary(numpy=False)
print("NGAFID_Dataset_Manager initialized and data dictionary constructed.")

# Restore the original download method (good practice)
NGAFID_Dataset_Manager.download = _original_download_method

Dataset '2days' found at /content/2days.
Initializing NGAFID_Dataset_Manager with dataset '2days'...
NGAFID_Dataset_Manager's internal download method bypassed as data exists.


  0%|          | 0/11446 [00:00<?, ?it/s]

NGAFID_Dataset_Manager initialized and data dictionary constructed.


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
number_classes = len(dm.flight_header_df['class'].unique())
number_classes

number_hierarchies = len(dm.flight_header_df['hclass'].unique())
number_hierarchies


5

# DEFINE TASK

In [20]:
mode = 'before_after'
# mode = 'hierarchy_basic'
#mode = 'both'
# mode = 'classes'

# DETERMINE MODEL STRUCTURE BASED ON OUTPUTS
two_output = False
if mode == 'before_after':
    nb_classes = 1
elif mode == 'hierarchy_basic':
    nb_classes = number_hierarchies + 1
    two_output = True
elif mode == 'both':
    two_output = True
    nb_classes = number_classes+1
else:
    nb_classes = number_classes+1

# TRAIN MODEL

In [21]:
import tensorflow as tf

class Saver(tf.keras.callbacks.Callback):

    def __init__(self, model_name):
        super().__init__()
        self.model_name = model_name
        # self.bucket = bucket
        # self.fs = gcsfs.GCSFileSystem(project='tpu-44747', token = 'gckey.json')
        self.start = 5
        self.best = 0

    def on_epoch_end(self, epoch, logs=None):
        if epoch > self.start:
            pass

        metric = 'val_loss'

        try:
            if logs.get(metric) >= 0.85 and logs.get(metric) > self.best:
                print('saving good model')
                lpath = self.model_name + '.h5'
                self.model.save(lpath, include_optimizer=False)
                self.best = logs.get(metric)
        except Exception as E:
            print('encountered some error in model saving process')
            print(E)
            pass

In [1]:
from tqdm.autonotebook import tqdm

# Removed redundant definitions for dm, dataset_name, destination_base_path, and NGAFID_Dataset_Manager import.
# These are now expected to be initialized by a preceding cell (e.g., kDYNIjFdtMEb).
import pandas as pd # Keep pandas import as it's used later

model_name = 'ITIME_BINARY'
# model_name = 'CONVMHSA_BOTH'
save_path = ''


flight_res = []


for fold in tqdm(range(5)):

    save_filename = save_path + '%s_%i' % (model_name, fold)


    train_ds = dm.get_tf_dataset(fold = fold, training = True, shuffle = 1000, repeat = True, mode = mode, batch_size = 128)
    test_ds = dm.get_tf_dataset(fold = fold, training  = False, shuffle = False, repeat = False, mode = mode, batch_size = 128)


    with strategy.scope():
        model = Classifier_INCEPTION(input_shape = (4096, 23), nb_classes=nb_classes, two_output = two_output ).model
        lr = 1e-4

        # model = get_mhsa_model(nbclass=nb_classes, mode = mode)
        # lr = 3e-5

        if two_output:

            model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=lr),
                        metrics =
                                    {'target_class' : [tf.keras.metrics.SparseCategoricalAccuracy(name = 'multi_acc')],
                                    'before_after' : [tf.keras.metrics.BinaryAccuracy(name = 'single_acc')],
                                    }

                        ,
                        loss = {'target_class' : tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False, name = 'multi_loss'),
                                'before_after' : tf.keras.losses.BinaryCrossentropy(from_logits=False, name = 'single_loss')}
            )


        elif mode == 'before_after':

            model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=lr),
                        metrics = [
                                    tf.keras.metrics.BinaryAccuracy()
                        ],
                        loss = tf.keras.losses.BinaryCrossentropy(from_logits=False))


        elif mode == 'classes':

            model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=lr),
                        metrics = [
                                    tf.keras.metrics.SparseCategoricalAccuracy(name = 'multi_acc')
                        ],
                        loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False))

        else:
            raise


        callbacks = [
                tf.keras.callbacks.ModelCheckpoint(
                    filepath="best.weights.h5", # Changed from "best.h5" to "best.weights.h5"
                    save_best_only=True,  # Only save a model if `val_loss` has improved.
                    monitor="val_loss",
                    verbose=1,
                    save_weights_only=True
                )
            ]


        history = model.fit(
            train_ds,
            steps_per_epoch = 100,
            epochs = 200,
            validation_data = test_ds,
            callbacks = callbacks,
            verbose = True,

        )

        history = pd.DataFrame(history.history)
        history['epoch'] = history.index
        history['model'] = model_name
        history.to_csv(save_filename)

        model.load_weights('best.weights.h5') # Changed from "best.h5" to "best.weights.h5"


        flights_before_acc = {}

        for day in range(5):


            indices = list(dm.flight_header_df[ (dm.flight_header_df['number_flights_before'] == day) & (dm.flight_header_df.before_after == 1) & (dm.flight_header_df.fold == fold) & (dm.flight_header_df.label == 'intake gasket leak/damage')].index)
            print(len(indices))
            ds = tf.data.Dataset.from_tensor_slices(to_dict_of_list([example for example in dm.data_dict if example['id'] in indices]))

            ds = dm.get_tf_dataset(ds = ds , fold = 0, training  = False, shuffle = False, repeat = False, mode = mode, batch_size = 2)

            res = model.evaluate(ds)

            flights_before_acc[day] = res[-1]

        flight_res.append(flights_before_acc)

/tmp/ipykernel_13226/1640967196.py:1: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


  0%|          | 0/5 [00:00<?, ?it/s]

NameError: name 'dm' is not defined

In [ ]:
flight_res

In [ ]:
print(flight_res)